# Analisis de Sentimientos en reseñas de películas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/7-sentiment-analysis.ipynb)

Ahora pongamos en práctica algunos de estos conceptos en un caso más real. Para esta práctica vamos a hacer un análisis de sentimientos sobre unas reseñas de películas. Este caso sería una simple clasificación binaria y podemos utilizar cualquier modelo para ese fin, lo adicional aquí es el pre-procesamiento de las entradas de texto.

### Referencias
* [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action)

In [46]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [47]:
# !test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt

In [48]:
# !test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion1/moviereviews.tsv

Empecemos por cargar el dataset:

In [49]:
import pandas as pd
import numpy as np

reviews = pd.read_csv('./IMDB_Dataset.csv')

print("--- Primeras 5 filas ---")
print(reviews.head())

# 2. Revisar tipos de datos y nulos
print("\n\n--- Información del DataFrame ---")
reviews.info()

print("\n\n--- Distribución de Sentimientos ---")
print(reviews['sentiment'].value_counts())

--- Primeras 5 filas ---
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


--- Información del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


--- Distribución de Sentimientos ---
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


Se confirman 50mil reseñas, sin valores nulos y con un dataset perfectamente balanceado entre reseñas positivas y negativas.

In [50]:
reviews.dropna(inplace=True)
reviews.review = reviews.review.apply(lambda r: r.strip())
blanks = reviews[reviews.review == ''].index
reviews.drop(blanks, inplace=True)

In [51]:
reviews[reviews.review == ''].index

Index([], dtype='int64')

In [52]:
reviews.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Tenemos un dataset balanceado de 25mil ejemplares para cada clase después de eliminar los valores nulos (en caso de que existan)

Para hacer las cosas simples, vamos a utilizar un VADER para computar el puntaje de positivo o negativo. Este modelo ya viene implementado dentro de NLTK.

In [53]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\andres.borreroc\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [54]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()
reviews['scores'] = reviews.review.apply(lambda r: sid.polarity_scores(r))
reviews.head()

,review,sentiment,scores
0,One of the other reviewers has mentioned that ...,positive,"{'neg': 0.203, 'neu': 0.748, 'pos': 0.048, 'co..."
1,A wonderful little production. <br /><br />The...,positive,"{'neg': 0.053, 'neu': 0.776, 'pos': 0.172, 'co..."
2,I thought this was a wonderful way to spend ti...,positive,"{'neg': 0.094, 'neu': 0.714, 'pos': 0.192, 'co..."
3,Basically there's a family where a little boy ...,negative,"{'neg': 0.138, 'neu': 0.797, 'pos': 0.065, 'co..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"{'neg': 0.052, 'neu': 0.801, 'pos': 0.147, 'co..."


Con estos puntajes ahora podemos convertir el resultado en una etiqueta de predicción:

In [55]:
reviews['compound'] = reviews.scores.apply(lambda s: s['compound'])    
reviews['prediction'] = reviews['compound'].apply(lambda c: 'positive' if c > 0 else 'negative')
reviews.head()

,review,sentiment,scores,compound,prediction
0,One of the other reviewers has mentioned that ...,positive,"{'neg': 0.203, 'neu': 0.748, 'pos': 0.048, 'co...",-0.9951,negative
1,A wonderful little production. <br /><br />The...,positive,"{'neg': 0.053, 'neu': 0.776, 'pos': 0.172, 'co...",0.9641,positive
2,I thought this was a wonderful way to spend ti...,positive,"{'neg': 0.094, 'neu': 0.714, 'pos': 0.192, 'co...",0.9605,positive
3,Basically there's a family where a little boy ...,negative,"{'neg': 0.138, 'neu': 0.797, 'pos': 0.065, 'co...",-0.9213,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"{'neg': 0.052, 'neu': 0.801, 'pos': 0.147, 'co...",0.9744,positive


Y finalmente computar unas cuantas métricas de calidad del modelo:

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# Let's assume your DataFrame is named 'df' as in the previous guide
# And it has columns: 'review', 'sentiment', and 'predicted_sentiment'

# Mask for a random 10% of the data
# Note: np.random.rand() > 0.90 selects roughly 10% of the data
# mask = np.random.rand(len(reviews)) > 0.90
# df_sample = reviews[mask]

# --- CORRECT DEFINITION OF y_true and y_pred ---
# y_true should be the GROUND TRUTH labels
y_true = reviews.sentiment.values

# y_pred should be the model's PREDICTED labels
y_pred = reviews.prediction.values


# --- Now the metrics will work correctly ---
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred)


print(f"Accuracy:\n{acc:.2%}\n")
print(f"Classification Report:\n{cr}")
print(f"Confusion Matrix:\n{cm}")

Accuracy:
69.63%

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.54      0.64     25000
    positive       0.65      0.86      0.74     25000

    accuracy                           0.70     50000
   macro avg       0.72      0.70      0.69     50000
weighted avg       0.72      0.70      0.69     50000

Confusion Matrix:
[[13427 11573]
 [ 3611 21389]]


## Conclusiones del Análisis de Sentimiento
1. Rendimiento General del Modelo
Tras aplicar el modelo de análisis de sentimiento VADER sobre el dataset de 50,000 reseñas de IMDb, obtuve una precisión (accuracy) general del 69.63%. Este resultado es significativamente superior al baseline del 50% que se obtendría al clasificar las reseñas de forma aleatoria. Esto demuestra que el modelo fue capaz de identificar patrones lingüísticos y léxicos en el texto que se correlacionan efectivamente con un sentimiento positivo o negativo, validando su utilidad para esta tarea.

2. Análisis del Comportamiento del Modelo
Al analizar el reporte de clasificación y la matriz de confusión, identifiqué un comportamiento asimétrico en el rendimiento del modelo:

Detección de Reseñas Positivas: El modelo exhibió su mayor fortaleza en la identificación de reseñas positivas, alcanzando un recall de 0.86. Esto indica que cuando una reseña era genuinamente positiva, el modelo lograba capturarla correctamente en la gran mayoría de los casos. Sin embargo, su precisión para esta clase fue de solo 0.65, lo que se debió a una tendencia a clasificar erróneamente un número considerable de reseñas negativas como si fueran positivas (11,573 falsos positivos).

Detección de Reseñas Negativas: Por el contrario, el modelo demostró ser más conservador y fiable al predecir la negatividad. Con una precisión de 0.79, cuando el modelo etiquetaba una reseña como negativa, acertaba con bastante frecuencia. No obstante, su principal área de mejora radica en el recall de 0.54 para esta clase, lo que significa que el modelo no logró identificar casi la mitad de todas las reseñas que eran realmente negativas.

3. Conclusión Final y Próximos Pasos
En conclusión, el modelo VADER funciona como un clasificador de sentimiento viable y sustancialmente mejor que una conjetura aleatoria. Su principal característica es una alta sensibilidad para detectar sentimientos positivos, aunque esto viene a costa de una tendencia a ser "demasiado optimista", generando muchos falsos positivos.

